In [4]:
import pandas as pd
import numpy as np

In [11]:
def load_data():
    print("Loading...")
    
    data_folder = "UCI HAR Dataset/"

    # Loding Columns Names
    features = pd.read_csv(data_folder + 'features.txt', sep='\s+', header=None, names=['id', 'name'])
    columnNames = features['name'].values

    labelsMap = pd.read_csv(data_folder + 'activity_labels.txt', sep='\s+', header=None, names=['id', 'activity_name'])

    def read_set(set_name):
        X = pd.read_csv(f'{data_folder}{set_name}/X_{set_name}.txt', sep='\s+', header=None)
        X.columns = columnNames
        
        y = pd.read_csv(f'{data_folder}{set_name}/y_{set_name}.txt', sep='\s+', header=None, names=['Activity_ID'])
        
        # Mapping for readability
        y['Activity_Name'] = y['Activity_ID'].map(labelsMap.set_index('id')['activity_name'])
        subject = pd.read_csv(f'{data_folder}{set_name}/subject_{set_name}.txt', sep='\s+', header=None, names=['Subject_ID'])
        return pd.concat([subject, y, X], axis=1)

    train_df = read_set('train')
    test_df = read_set('test')

    print("")
    print(f"Training data {train_df.shape} : Test data {test_df.shape}")
    
    return train_df, test_df

In [ ]:
# this function for next phases contains 3D data 
def load_inertial_signals(set_name):
    print(f"Loading Raw Inertial Signals for {set_name}...")
    data_folder = f"UCI HAR Dataset/{set_name}/Inertial Signals/"
    
    filenames = [
        'body_acc_x_', 'body_acc_y_', 'body_acc_z_',
        'body_gyro_x_', 'body_gyro_y_', 'body_gyro_z_',
        'total_acc_x_', 'total_acc_y_', 'total_acc_z_'
    ]
    
    signals = []
    for name in filenames:
        file_path = f"{data_folder}{name}{set_name}.txt"
        data = pd.read_csv(file_path, sep='\s+', header=None)
        signals.append(data.values)

    return np.transpose(np.array(signals), (1, 2, 0))

X_train_raw = load_inertial_signals('train')
print(f"Raw Data Shape: {X_train_raw.shape}")

Loading Raw Inertial Signals for train...
Raw Data Shape: (7352, 128, 9)


In [ ]:
train_data, test_data = load_data()

print("\n Sample ")
print(train_data[['Subject_ID', 'Activity_Name', 'tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y']].head())

# Data Check
print(train_data['Activity_Name'].value_counts())

Loading...

Training data (7352, 564) : Test data (2947, 564)

 Sample 
   Subject_ID Activity_Name  tBodyAcc-mean()-X  tBodyAcc-mean()-Y
0           1      STANDING           0.288585          -0.020294
1           1      STANDING           0.278419          -0.016411
2           1      STANDING           0.279653          -0.019467
3           1      STANDING           0.279174          -0.026201
4           1      STANDING           0.276629          -0.016570
Activity_Name
LAYING                1407
STANDING              1374
SITTING               1286
WALKING               1226
WALKING_UPSTAIRS      1073
WALKING_DOWNSTAIRS     986
Name: count, dtype: int64


In [ ]:
# 1. Check for NaN values in the raining set
nan_counts_train = train_data.isnull().sum().sum()
print(f"Total NaN values in Training Data: {nan_counts_train}")

# Check for NaN values in the Test set
nan_counts_test = test_data.isnull().sum().sum()
print(f"Total NaN values in Test Data: {nan_counts_test}")

#Check for any duplicated rows
duplicates = train_data.duplicated().sum()
print(f"Total duplicated rows in Training Data: {duplicates}")

Total NaN values in Training Data: 0
Total NaN values in Test Data: 0
Total duplicated rows in Training Data: 0
